In [92]:
import pandas as pd

## Unemployment Data (Averaging the unemployment rates)

In [93]:
unemployment = pd.read_excel("Data/Unemployment_rate.xlsx", dtype={'State FIPS Code': str, 'County FIPS Code': str}, header=0)

In [94]:
unemployment.head()

,LAUS Code,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Labor Force,Employed,Unemployed,Unemploy-ment Rate (%)
0,CN0100100000000,01,001,"Autauga County, AL",Oct-24,28609,27798,811,2.8
1,CN0100300000000,01,003,"Baldwin County, AL",Oct-24,117918,114611,3307,2.8
2,CN0100500000000,01,005,"Barbour County, AL",Oct-24,8825,8451,374,4.2
3,CN0100700000000,01,007,"Bibb County, AL",Oct-24,8727,8447,280,3.2
4,CN0100900000000,01,009,"Blount County, AL",Oct-24,27134,26401,733,2.7


In [95]:
unemployment = unemployment.drop(["LAUS Code", "Labor Force", "Employed", "Unemployed"], axis=1)

In [96]:
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1
45094,NaN,NaN,NaN,NaN,"SOURCE: BLS, LAUS"
45095,NaN,NaN,NaN,NaN,"January 16, 2026"


In [97]:
unemployment = unemployment.drop([45094, 45095])
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45089,72,145,"Vega Baja Municipio, PR",Nov-25 p,5.5
45090,72,147,"Vieques Municipio, PR",Nov-25 p,4.9
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1


In [98]:
unemployment.dtypes

State FIPS Code                   object
County FIPS Code                  object
County Name/State Abbreviation    object
Period                            object
Unemploy-ment Rate (%)            object
dtype: object

In [99]:
# 1. Force everything to numeric. The '–' for all oct 25 entries will become NaN (null)
unemployment["Unemploy-ment Rate (%)"] = pd.to_numeric(unemployment["Unemploy-ment Rate (%)"], errors='coerce')

# 2. Drop the rows that are now NaN
unemployment = unemployment.dropna(subset=["Unemploy-ment Rate (%)"])

unemployment['full_fips'] = (
    unemployment['State FIPS Code'].astype(str).str.zfill(2) + 
    unemployment['County FIPS Code'].astype(str).str.zfill(3)
)

# 3. Now convert to int
unemployment["Unemploy-ment Rate (%)"] = unemployment["Unemploy-ment Rate (%)"].astype(int)

In [100]:
unemployment = unemployment.groupby(["full_fips", "County Name/State Abbreviation"]).agg(unemployment_rate = ("Unemploy-ment Rate (%)", "mean")).reset_index()

In [101]:
unemployment.tail()

,full_fips,County Name/State Abbreviation,unemployment_rate
3216,72145,"Vega Baja Municipio, PR",4.857143
3217,72147,"Vieques Municipio, PR",4.357143
3218,72149,"Villalba Municipio, PR",8.285714
3219,72151,"Yabucoa Municipio, PR",8.214286
3220,72153,"Yauco Municipio, PR",9.428571


In [102]:
state_map = {
    # 50 States
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    
    # District & Territories
    'DC': 'District of Columbia',
    'PR': 'Puerto Rico',
    'VI': 'Virgin Islands',
    'GU': 'Guam',
    'AS': 'American Samoa',
    'MP': 'Northern Mariana Islands'
}

In [103]:
# Split the column into two temporary parts
# n=1 ensures we only split at the last comma
split_data = unemployment['County Name/State Abbreviation'].str.split(',', n=1, expand=True)

# 2. Clean up the abbreviation (remove spaces)
# We use the 'state_map' dictionary we created earlier
state_abbr = split_data[1].str.strip().str.upper()
state_full = state_abbr.map(state_map)

# 3. Create the new combined column
# split_data[0] is the County Name
unemployment['County Name/State'] = split_data[0] + ", " + state_full

In [104]:
unemployment.head()

,full_fips,County Name/State Abbreviation,unemployment_rate,County Name/State
0,01001,"Autauga County, AL",2.153846,"Autauga County, Alabama"
1,01003,"Baldwin County, AL",2.307692,"Baldwin County, Alabama"
2,01005,"Barbour County, AL",3.538462,"Barbour County, Alabama"
3,01007,"Bibb County, AL",2.538462,"Bibb County, Alabama"
4,01009,"Blount County, AL",2.230769,"Blount County, Alabama"


In [105]:
unemployment["County Name/State"].isna().sum()

1

In [106]:
missingrows = unemployment[unemployment['County Name/State'].isna()]
print(missingrows)

    full_fips County Name/State Abbreviation  unemployment_rate  \
321     11001           District of Columbia           5.384615   

    County Name/State  
321               NaN  


In [107]:
# df.loc[condition, column] = value
unemployment.loc[unemployment['County Name/State Abbreviation'] == 'District of Columbia', 'County Name/State'] = 'District of Columbia, District of Columbia'

In [108]:
unemployment_final = unemployment.drop("County Name/State Abbreviation", axis=1)

unemployment_final['unemployment_rate'] = unemployment_final['unemployment_rate']/100

In [109]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State
3216,72145,0.048571,"Vega Baja Municipio, Puerto Rico"
3217,72147,0.043571,"Vieques Municipio, Puerto Rico"
3218,72149,0.082857,"Villalba Municipio, Puerto Rico"
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico"
3220,72153,0.094286,"Yauco Municipio, Puerto Rico"


In [110]:
unemployment_final["County Name/State"].isna().sum()

0

## Poverty Rate (dublicate rows and a few calculated columns)

In [111]:
poverty = pd.read_csv("Data/Poverty_rate.csv", header=0)

In [112]:
poverty.head()

,County,Population,Poverty,Poverty_Ugstudents,Unnamed: 4
0,"Autauga County, Alabama",NaN,NaN,NaN,NaN
1,Estimate,"57,409","6,473",134,NaN
2,"Baldwin County, Alabama",NaN,NaN,NaN,NaN
3,Estimate,"237,007","23,381",995,NaN
4,"Barbour County, Alabama",NaN,NaN,NaN,NaN


In [113]:
poverty = poverty.drop("Unnamed: 4", axis=1)

In [114]:
# bfill(axis=0) pulls data UP from the row below
poverty = poverty.bfill(axis=0, limit=1)

# Filter out the estimate rows
poverty = poverty[~poverty['County'].str.contains('estimate', case=False, na=False)]

In [115]:
poverty = poverty.reset_index()

In [116]:
poverty.tail()

,index,County,Population,Poverty,Poverty_Ugstudents
3217,6434,"Vega Baja Municipio, Puerto Rico","52,600","21,470",841
3218,6436,"Vieques Municipio, Puerto Rico","7,923","4,619",104
3219,6438,"Villalba Municipio, Puerto Rico","20,981","7,972",308
3220,6440,"Yabucoa Municipio, Puerto Rico","28,813","13,615",610
3221,6442,"Yauco Municipio, Puerto Rico","32,129","13,508",444


In [117]:
for i in ['Population', 'Poverty', 'Poverty_Ugstudents']:
    poverty[i] = pd.to_numeric(poverty[i].str.replace(',', ''), errors='coerce')

In [118]:
poverty['poverty_rate'] = (poverty['Poverty'] - poverty['Poverty_Ugstudents'])/poverty['Population']

In [119]:
poverty_final = poverty.drop(["Population", "Poverty", 'Poverty_Ugstudents'], axis = 1)

In [120]:
poverty_final.tail()

,index,County,poverty_rate
3217,6434,"Vega Baja Municipio, Puerto Rico",0.392186
3218,6436,"Vieques Municipio, Puerto Rico",0.569860
3219,6438,"Villalba Municipio, Puerto Rico",0.365283
3220,6440,"Yabucoa Municipio, Puerto Rico",0.451359
3221,6442,"Yauco Municipio, Puerto Rico",0.406611


In [121]:
poverty_final["County"].isna().sum()

0

### Checking for mismatches with county names in poverty and unemployment dfs as county names is the only common column to join them

In [122]:
missing_county =  set(unemployment_final['County Name/State']) - set(poverty['County'])
print(missing_county)

{'Juana Diaz Municipio, Puerto Rico', 'Comerio Municipio, Puerto Rico', 'Las Marias Municipio, Puerto Rico', 'Rio Grande Municipio, Puerto Rico', 'Guanica Municipio, Puerto Rico', 'Denver County/city, Colorado', 'Broomfield County/city, Colorado', 'Canovanas Municipio, Puerto Rico', 'Penuelas Municipio, Puerto Rico', 'Manati Municipio, Puerto Rico', 'Catano Municipio, Puerto Rico', 'Bayamon Municipio, Puerto Rico', 'Yakutat Borough/city, Alaska', 'Anasco Municipio, Puerto Rico', 'Honolulu County/city, Hawaii', 'San German Municipio, Puerto Rico', 'Dona Ana County, New Mexico', 'Juneau Borough/city, Alaska', 'Nantucket County/town, Massachusetts', 'Philadelphia County/city, Pennsylvania', 'Mayaguez Municipio, Puerto Rico', 'Loiza Municipio, Puerto Rico', 'Wrangell Borough/city, Alaska', 'San Sebastian Municipio, Puerto Rico', 'Rincon Municipio, Puerto Rico', 'Sitka Borough/city, Alaska', 'Anchorage Borough/municipality, Alaska', 'San Francisco County/city, California'}


In [123]:
missing_county =  set(poverty['County']) - set(unemployment_final['County Name/State'])
print(missing_county)

{'Comerío Municipio, Puerto Rico', 'Rincón Municipio, Puerto Rico', 'Kalawao County, Hawaii', 'San Francisco County, California', 'Añasco Municipio, Puerto Rico', 'Manatí Municipio, Puerto Rico', 'Río Grande Municipio, Puerto Rico', 'Canóvanas Municipio, Puerto Rico', 'Nantucket County, Massachusetts', 'Honolulu County, Hawaii', 'San Germán Municipio, Puerto Rico', 'Las Marías Municipio, Puerto Rico', 'Juneau City and Borough, Alaska', 'Anchorage Municipality, Alaska', 'Sitka City and Borough, Alaska', 'Wrangell City and Borough, Alaska', 'Mayagüez Municipio, Puerto Rico', 'Broomfield County, Colorado', 'Cataño Municipio, Puerto Rico', 'Guánica Municipio, Puerto Rico', 'Doña Ana County, New Mexico', 'Juana Díaz Municipio, Puerto Rico', 'Philadelphia County, Pennsylvania', 'Denver County, Colorado', 'Loíza Municipio, Puerto Rico', 'Bayamón Municipio, Puerto Rico', 'San Sebastián Municipio, Puerto Rico', 'Yakutat City and Borough, Alaska', 'Peñuelas Municipio, Puerto Rico'}


In [124]:
import unicodedata

def make_common_key(text):
    if pd.isna(text): return "nan"
    
    # Lowercase and remove punctuation
    text = str(text).lower().replace(',', '').replace('/', ' ')
    
    # Strip Accents (turns ñ to n, í to i)
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode("utf-8")
    
    # Strip "Noise" words that differ between your tables
    noise_words = [
        "municipio", "municipality", "city and borough", "borough/city", 
        "county/city", "county/town", "county", "borough", "town", "city"
    ]
    for word in noise_words:
        text = text.replace(word, "")
    
    # Remove extra spaces
    return " ".join(text.split())

In [125]:
# Apply to both DataFrames
unemployment_final['common_key'] = unemployment_final['County Name/State'].apply(make_common_key)
poverty_final['common_key'] = poverty_final['County'].apply(make_common_key)

In [126]:
missing_county =  set(unemployment_final['common_key']) - set(poverty_final['common_key'])
print(missing_county)

set()


In [127]:
missing_county =  set(poverty_final['common_key']) - set(unemployment_final['common_key']) 
print(missing_county)

{'kalawao hawaii'}


In [128]:
# Select rows where county is kalawao county, hawaii to check if it exists
filtered_df = poverty_final[poverty_final['County'] == 'Kalawao County, Hawaii']
filtered_df.head()

,index,County,poverty_rate,common_key
550,1100,"Kalawao County, Hawaii",0.227273,kalawao hawaii


In [129]:
# Select rows where county is maui county to check its unemployment rate as kalawao is mixed with that county
filtered_df2 = unemployment_final[unemployment_final['County Name/State'] == 'Maui County, Hawaii']
filtered_df2.head()

,full_fips,unemployment_rate,County Name/State,common_key
551,15009,0.025385,"Maui County, Hawaii",maui hawaii


In [130]:
# Add the respective row to unemployment_df, the fips code is 15005

# Define the values for Kalawao County
# Using Maui County's rate as the proxy per BLS/Census standard
new_row = {
    'full_fips': '15005',
    'unemployment_rate': 2.538462,
    'County Name/State': 'Kalawao County, Hawaii',
    'common_key': 'kalawao hawaii'
}

# Insert into the DataFrame
# loc[len(df)] appends it to the very bottom
unemployment_final.loc[len(unemployment_final)] = new_row

In [131]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State,common_key
3217,72147,0.043571,"Vieques Municipio, Puerto Rico",vieques puerto rico
3218,72149,0.082857,"Villalba Municipio, Puerto Rico",villalba puerto rico
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico",yabucoa puerto rico
3220,72153,0.094286,"Yauco Municipio, Puerto Rico",yauco puerto rico
3221,15005,2.538462,"Kalawao County, Hawaii",kalawao hawaii


## Disability rate

In [132]:
disability = pd.read_csv("Data/Disability_rate.csv", header=0)

In [133]:
disability.head(10)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
1,Total,NaN
2,Estimate,"58,355"
3,With a disability,NaN
4,Estimate,"9,473"
5,Percent with a disability,NaN
6,Estimate,16.20%
7,"Baldwin County, Alabama",NaN
8,Total,NaN
9,Estimate,"243,859"


In [134]:
# bfill(axis=0) pulls data UP from the row below
disability = disability.bfill(axis=0, limit=1)

# Filter out the estimate rows
disability = disability[~disability['County'].str.contains('estimate', case=False, na=False)]
disability = disability[~disability['County'].str.contains('total', case=False, na=False)]
# Keep rows where the County is NOT exactly "with a disability", we did not use "str.contains" because in that case "percent with a disability" also gets deleted
disability = disability[disability['County'].str.strip() != 'With a disability']

In [135]:
disability.head(5)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
5,Percent with a disability,16.20%
7,"Baldwin County, Alabama",NaN
12,Percent with a disability,13.20%
14,"Barbour County, Alabama",NaN


In [136]:
# Getting the percentages aligned to the county names
disability = disability.bfill(axis=0, limit=1)

# Removing the redundant rows
disability = disability[disability['County'].str.strip() != 'Percent with a disability']

# renaming column name
disability.columns = ['County', 'disability_rate']

In [137]:
disability.dtypes

County             object
disability_rate    object
dtype: object

In [138]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",21.90%
22526,"Vieques Municipio, Puerto Rico",13.60%
22533,"Villalba Municipio, Puerto Rico",22.90%
22540,"Yabucoa Municipio, Puerto Rico",12.80%
22547,"Yauco Municipio, Puerto Rico",27.60%


In [139]:
disability['disability_rate'] = pd.to_numeric(disability['disability_rate'].astype(str).str.replace('%', ''), errors='coerce')
disability['disability_rate'] = disability['disability_rate']/100

In [140]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",0.219
22526,"Vieques Municipio, Puerto Rico",0.136
22533,"Villalba Municipio, Puerto Rico",0.229
22540,"Yabucoa Municipio, Puerto Rico",0.128
22547,"Yauco Municipio, Puerto Rico",0.276


In [141]:
disability["County"].isna().sum()

0

In [142]:
# Use the predefined function to make the county names same for all datasets
disability['common_key'] = disability['County'].apply(make_common_key)

# resetting index
disability_final = disability.reset_index()

In [143]:
disability_final.tail() 

,index,County,disability_rate,common_key
3217,22519,"Vega Baja Municipio, Puerto Rico",0.219,vega baja puerto rico
3218,22526,"Vieques Municipio, Puerto Rico",0.136,vieques puerto rico
3219,22533,"Villalba Municipio, Puerto Rico",0.229,villalba puerto rico
3220,22540,"Yabucoa Municipio, Puerto Rico",0.128,yabucoa puerto rico
3221,22547,"Yauco Municipio, Puerto Rico",0.276,yauco puerto rico


## Homeownership rate

In [144]:
homeownership = pd.read_csv("Data/Homeownership_rate.csv", header=0)

In [145]:
homeownership.tail()

,Label (Grouping),HOUSING TENURE!!Occupied housing units!!Owner-occupied
9661,Estimate,"8,597"
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9664,Estimate,"9,653"
9665,Percent,73.0%


In [146]:
homeownership.columns = ["County", "homeownership_rate"]

In [147]:
homeownership = homeownership[homeownership["County"].str.strip() != "Estimate"]

In [148]:
homeownership.tail()

,County,homeownership_rate
9659,Percent,77.9%
9660,"Yabucoa Municipio, Puerto Rico",NaN
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9665,Percent,73.0%


In [149]:
homeownership = homeownership.bfill(axis=0, limit=1)

homeownership = homeownership[homeownership["County"].str.strip() != "Percent"]

homeownership_final = homeownership.reset_index()

In [150]:
homeownership_final.tail()

,index,County,homeownership_rate
3217,9651,"Vega Baja Municipio, Puerto Rico",74.8%
3218,9654,"Vieques Municipio, Puerto Rico",66.8%
3219,9657,"Villalba Municipio, Puerto Rico",77.9%
3220,9660,"Yabucoa Municipio, Puerto Rico",70.7%
3221,9663,"Yauco Municipio, Puerto Rico",73.0%


In [151]:
homeownership_final["County"].isna().sum()

0

In [152]:
homeownership_final['common_key'] = homeownership_final['County'].apply(make_common_key)

In [153]:
homeownership_final.head()

,index,County,homeownership_rate,common_key
0,0,"Autauga County, Alabama",77.1%,autauga alabama
1,3,"Baldwin County, Alabama",77.6%,baldwin alabama
2,6,"Barbour County, Alabama",68.2%,barbour alabama
3,9,"Bibb County, Alabama",79.2%,bibb alabama
4,12,"Blount County, Alabama",81.0%,blount alabama
